In [1]:
!rm -rf /content/yolov1-cupy
!git clone https://github.com/mihnea-popescu/yolov1-cupy.git

import sys
sys.path.append('/content/yolov1-cupy')

from main import TestClass

test = TestClass()
test.test()          # IT'S WORKING!

Cloning into 'yolov1-cupy'...
remote: Enumerating objects: 85, done.
remote: Counting objects: 100% (85/85), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 85 (delta 34), reused 63 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (85/85), 51.32 KiB | 1.90 MiB/s, done.
Resolving deltas: 100% (34/34), done.
Github classes initialized!


Downloading the ImageNet10 dataset from kaggle

In [2]:
!curl -L -o /content/imagenet10.zip https://www.kaggle.com/api/v1/datasets/download/liusha249/imagenet10
print("Extracting dataset (quiet unzip)...")
!unzip -q /content/imagenet10.zip -d /content/yolov1-cupy
print("Done.")
!rm /content/imagenet10.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 1304M  100 1304M    0     0   149M      0  0:00:08  0:00:08 --:--:--  162M
Extracting dataset (quiet unzip)...
Done.


## Process Images

In [3]:
from image_batch_loader import image_label_batch

batch_size = 8
x, y_cpu = image_label_batch("/content/yolov1-cupy", batch_size=batch_size)
print("x.shape:", x.shape, "dtype:", x.dtype)
print("labels:", y_cpu)

x.shape: (8, 3, 224, 224) dtype: float64
labels: [8 2 6 9 8 5 3 0]


## Mini Darknet Structure

| Block | Layers                                                           | Output size   |
|-------|------------------------------------------------------------------|--------------|
| Input | —                                                                | 3 × 224 × 224|
| 1     | Conv2D(3→16, 3, pad=1) → BN → LeakyReLU → MaxPool(2)            | 16 × 112 × 112|
| 2     | Conv2D(16→32, 3, pad=1) → BN → LeakyReLU → MaxPool(2)           | 32 × 56 × 56 |
| 3     | Conv2D(32→64, 3, pad=1) → BN → LeakyReLU → MaxPool(2)           | 64 × 28 × 28 |
| 4     | Conv2D(64→128, 3, pad=1) → BN → LeakyReLU → MaxPool(2)          | 128 × 14 × 14|
| 5     | Conv2D(128→256, 3, pad=1) → BN → LeakyReLU → MaxPool(2)         | 256 × 7 × 7  |
| Head  | AvgPool(7) → Flatten → Linear(256, 10)                          | 10           |

In [6]:
from mini_darknet import MiniDarknet

model = MiniDarknet(num_classes=10)
logits = model.forward(x)
print("logits.shape:", logits.shape, "dtype:", logits.dtype)
print(logits)

logits.shape: (8, 10) dtype: float64
[[-0.15225991 -0.64359484  0.324078    0.25361378 -0.74759098  0.57127184
  -0.64728793 -0.26471855 -0.44988212 -1.11433605]
 [-0.08309234 -0.174201    0.42458892  0.13973613 -0.37561853  0.12187017
  -0.24893515 -0.25524047 -0.48270584 -0.12698167]
 [-0.31201739 -0.45635979  0.08723442  0.10114042 -0.71328384  0.40886404
  -0.41951316 -0.17995174 -0.30051539 -0.87007652]
 [-0.58350715 -0.57031059 -0.53892143  0.01831552 -1.30635587  0.40212595
  -0.53236125 -0.09646591  0.21503369 -1.8930821 ]
 [-0.06996316 -0.2501856   0.29879143  0.38044324 -0.78363114  0.2901125
  -0.23641147 -0.04904887 -0.1839002  -0.75853872]
 [-0.07249255 -0.14011595 -0.09318888  0.24040455 -0.7055128   0.4665175
  -0.33112196 -0.16388183 -0.23037942 -0.74525787]
 [-0.03513397 -0.13149272  0.48907309  0.27623153 -0.599816    0.10912487
  -0.26298906 -0.14245058 -0.39110478 -0.31521956]
 [ 0.07401375 -0.52002384  0.2223135   0.2755862  -0.71655063  0.75687591
  -0.54398907 -0

In [ ]:
from softmax import softmax

probs = softmax(logits, axis=1)
print("probs.shape:", probs.shape)
print("sums along classes (should be ~1):", probs.sum(axis=1))